In [2]:
!pip install -q nltk gensim

In [3]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
import nltk
from nltk.corpus import stopwords
from gensim.models import Word2Vec

In [13]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aditi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aditi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aditi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
text = """
Alice was beginning to get very tired of sitting by her sister
on the bank, and of having nothing to do.

Alice was thinking about her journey when she saw a rabbit.
The rabbit was running quickly through the garden.

Alice followed the rabbit and entered a wonderful world.
She met many strange characters and had many adventures.

The queen was sitting in the garden.
Alice talked to the queen and asked many questions.
The rabbit said that the journey was very interesting.
"""
print(text[:500])


Alice was beginning to get very tired of sitting by her sister
on the bank, and of having nothing to do.

Alice was thinking about her journey when she saw a rabbit.
The rabbit was running quickly through the garden.

Alice followed the rabbit and entered a wonderful world.
She met many strange characters and had many adventures.

The queen was sitting in the garden.
Alice talked to the queen and asked many questions.
The rabbit said that the journey was very interesting.



In [15]:
# Convert to lowercase
text = text.lower()

# Remove digits
text = re.sub(r'\d+', '', text)

# Remove special characters and punctuation
text = re.sub(r'[^a-z\s]', ' ', text)

# Remove extra spaces
text = re.sub(r'\s+', ' ', text)

print(text[:500])

 alice was beginning to get very tired of sitting by her sister on the bank and of having nothing to do alice was thinking about her journey when she saw a rabbit the rabbit was running quickly through the garden alice followed the rabbit and entered a wonderful world she met many strange characters and had many adventures the queen was sitting in the garden alice talked to the queen and asked many questions the rabbit said that the journey was very interesting 


In [16]:
sentences = nltk.sent_tokenize(text)

tokenized_sentences = []

for sentence in sentences:
    words = nltk.word_tokenize(sentence)
    tokenized_sentences.append(words)

print(tokenized_sentences[:2])

[['alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank', 'and', 'of', 'having', 'nothing', 'to', 'do', 'alice', 'was', 'thinking', 'about', 'her', 'journey', 'when', 'she', 'saw', 'a', 'rabbit', 'the', 'rabbit', 'was', 'running', 'quickly', 'through', 'the', 'garden', 'alice', 'followed', 'the', 'rabbit', 'and', 'entered', 'a', 'wonderful', 'world', 'she', 'met', 'many', 'strange', 'characters', 'and', 'had', 'many', 'adventures', 'the', 'queen', 'was', 'sitting', 'in', 'the', 'garden', 'alice', 'talked', 'to', 'the', 'queen', 'and', 'asked', 'many', 'questions', 'the', 'rabbit', 'said', 'that', 'the', 'journey', 'was', 'very', 'interesting']]


In [17]:
stop_words = set(stopwords.words('english'))

cleaned_sentences = []

for sentence in tokenized_sentences:
    words = [word for word in sentence if word not in stop_words]
    if words:
        cleaned_sentences.append(words)

print(cleaned_sentences[:3])

[['alice', 'beginning', 'get', 'tired', 'sitting', 'sister', 'bank', 'nothing', 'alice', 'thinking', 'journey', 'saw', 'rabbit', 'rabbit', 'running', 'quickly', 'garden', 'alice', 'followed', 'rabbit', 'entered', 'wonderful', 'world', 'met', 'many', 'strange', 'characters', 'many', 'adventures', 'queen', 'sitting', 'garden', 'alice', 'talked', 'queen', 'asked', 'many', 'questions', 'rabbit', 'said', 'journey', 'interesting']]


In [18]:
model = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=100,
    window=2,
    min_count=2,
    sg=0,
    workers=4,
    epochs=10
)

print("CBOW model trained successfully.")

CBOW model trained successfully.


In [19]:
words = list(model.wv.index_to_key)

print("Vocabulary size:", len(words))
print("First 20 words:", words[:20])

Vocabulary size: 7
First 20 words: ['rabbit', 'alice', 'many', 'queen', 'garden', 'journey', 'sitting']


In [21]:
word = "alice"

if word in model.wv:
    print("Vector representation of", word)
    print(model.wv[word][:10])
else:
    print("Word not found")

Vector representation of alice
[-0.00861969  0.00366574  0.00518988  0.00574194  0.00746692 -0.00616768
  0.00110561  0.00604728 -0.00284005 -0.00617352]


In [22]:
word = "alice"

similar_words = model.wv.most_similar(word, topn=10)

print("Words similar to", word)

for w, score in similar_words:
    print(w, score)

Words similar to alice
queen 0.06797593086957932
journey 0.0093911811709404
garden 0.004503030329942703
rabbit -0.010839170776307583
many -0.02370821125805378
sitting -0.1141158938407898


In [23]:
word1 = "alice"
word2 = "queen"

if word1 in model.wv and word2 in model.wv:
    similarity = model.wv.similarity(word1, word2)
    print("Similarity between", word1, "and", word2, ":", similarity)
else:
    print("One or both words are not in vocabulary")

Similarity between alice and queen : 0.06797595


In [24]:
context_words = ["alice", "said"]

predictions = model.predict_output_word(
    context_words,
    topn=10
)

print("Predicted words using CBOW:")

if predictions:
    for word, score in predictions:
        print(word, score)
else:
    print("No predictions available")

Predicted words using CBOW:
rabbit 0.14285772
alice 0.14285772
garden 0.14285772
queen 0.1428575
journey 0.14285678
sitting 0.14285634
many 0.14285624


In [26]:

import numpy as np;
context_words = ["alice", "queen"]

vectors = [
    model.wv[word]
    for word in context_words
    if word in model.wv
]

if vectors:
    context_vector = np.mean(vectors, axis=0)

    result = model.wv.similar_by_vector(
        context_vector,
        topn=10
    )

    print("Predicted words:")

    for word, score in result:
        print(word, score)

Predicted words:
queen 0.7308298349380493
alice 0.7306597828865051
journey 0.09640675783157349
many -0.025458531454205513
garden -0.02745186723768711
sitting -0.04963584989309311
rabbit -0.08383447676897049
